In [1]:
import torch
import plotly.graph_objects as go
import dash
from dash import Dash, Patch, dcc, html, Input, Output, clientside_callback
from raytracer_2d import (
    get_rect_subdivs,
    get_rects_verts_2d,
    get_furthest_corners,
    get_convex_hull_2d,
    if_rects_in_hull_2d,
    if_rects_intersect_hull_2d,
    get_fov_pixel_centers_2d,
    get_rects_center_2d,
)

In [10]:
rect = torch.tensor(
    [[32, 4], [8, 0], [0, 8]],
    dtype=torch.float32,
)
xtal_nsubs = torch.tensor([9, 9])

rect_subs = get_rect_subdivs(rect, xtal_nsubs)

plate_rects = torch.tensor(
    [
        [[29, 8], [2, 0], [0, 8]],
        [[29, -2], [2, 0], [0, 8]],
    ],
    dtype=torch.float32,
)
# define the field of view (FOV)
fov_npx = torch.tensor([4, 4])
fov_mmppx = torch.tensor([4, 4])
fov_dims = fov_npx * fov_mmppx

fov_corners = torch.tensor(
    [
        [-fov_dims[0], -fov_dims[1]],
        [fov_dims[0], -fov_dims[1]],
        [fov_dims[0], fov_dims[1]],
        [-fov_dims[0], fov_dims[1]],
    ]
).view(-1, 2)

pa_tensor = get_fov_pixel_centers_2d(fov_npx, fov_mmppx)
pb_tensor = get_rects_center_2d(rect_subs)
rect_subs_verts = get_rects_verts_2d(rect_subs)


pixel_corners = get_furthest_corners(pa_tensor.view(-1, 2), 4)

i_list = torch.arange(rect_subs.shape[0])
intersect_rects_indices = []
inside_rects_indices = []
for i in i_list:
    candidates = torch.vstack(
        (plate_rects, rect_subs[~torch.arange(rect_subs.shape[0]).eq(i)])
    )
    candidates_verts = get_rects_verts_2d(candidates)
    hull = get_convex_hull_2d(torch.vstack((pb_tensor[i], pixel_corners)))
    valid_indices, test_rays = if_rects_intersect_hull_2d(
        candidates, hull, pb_tensor[i]
    )
    insid_indices = if_rects_in_hull_2d(candidates, hull)
    intersect_rects_indices.append(valid_indices.tolist())
    inside_rects_indices.append(torch.argwhere(insid_indices))


In [ ]:
fig = go.Figure()

fig.update_layout(
    # hovermode="x unified",
    # xaxis=dict(showspikes=True),
    # yaxis=dict(showspikes=True),
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis_scaleanchor="y",
    # yaxis_scaleanchor="x",
    plot_bgcolor="rgba(0,0,0,0)",
    paper_bgcolor="rgba(0,0,0,0)",
)

fig.add_trace(
    go.Scatter(
        x=fov_corners[[0, 1, 2, 3, 0], 0],
        y=fov_corners[[0, 1, 2, 3, 0], 1],
        mode="lines",
        line=dict(color="black"),
        name="FOV",
        showlegend=False,
        hoverinfo="name",
    )
)

plate_rects_verst = get_rects_verts_2d(plate_rects)
for i, verts in enumerate(plate_rects_verst):
    fig.add_trace(
        go.Scatter(
            x=verts[[0, 1, 2, 3, 0], 0],
            y=verts[[0, 1, 2, 3, 0], 1],
            mode="lines",
            line=dict(color="green", width=1),
            name=f"plate {i}",
            fill="toself",
            hoverinfo="name",
            showlegend=False,
        ),
    )

for i, verts in enumerate(rect_subs_verts):
    fig.add_trace(
        go.Scatter(
            x=verts[[0, 1, 2, 3, 0], 0],
            y=verts[[0, 1, 2, 3, 0], 1],
            mode="lines",
            line=dict(color="blue", width=1),
            name=f"{i}",
            # hoverinfo="name",
            fill="toself",
            fillcolor="rgba(0,0,255,0)",
            hoverinfo="none",
            showlegend=False,
        )
    )



# Initialize a JupyterDash app
app = Dash(__name__)

# Define the app layout
app.layout = html.Div(
    [html.Div(id="hover-output"), dcc.Graph(id="graph", figure=fig)]
)

# app.clientside_callback(
#     """
#     function(hoverData) {
#         if (hoverData) {
#             var polygon_id = hoverData.points[0].curveNumber;
#             return "Hovered polygon: " + polygon_id;
#         } else {
#             return "Hover over a point";
#         }
#     }
#     """,
#     Output("hover-output", "children"),
#     Input("graph", "hoverData"),
# )

# app.clientside_callback(
#     """
#     function(hoverData, figure) {
#         if (hoverData) {
#             id = hoverData.points[0].curveNumber;
#             var updated_ = figure;
#             updated_.data[id].fillcolor = "rgba(255,0,0,0.5)";
#             return updated_;
#         }
#         return figure;
#     }
#     """,
#     Output("graph", "figure"),
#     Input("graph", "hoverData"),
#     Input("graph", "figure"),
# )


@app.callback(
    Output("graph", "figure"),
    Output("hover-output", "children"),
    Input("graph", "hoverData"),
    Input("graph", "clickData"),
)
def highlight_hover(hoverData, clickData):
    # updated_fig = Patch()
    if hoverData:
        updated_fig = Patch()
        updated_fig.data = fig.data
        polygon_id = hoverData["points"][0]["curveNumber"]
        if polygon_id > 1:
            updated_fig.data[polygon_id].fillcolor = "rgba(255,0,0,0.5)"
            if clickData:
                polygon_id_click = clickData["points"][0]["curveNumber"]
                if polygon_id_click == polygon_id:
                    print(intersect_rects_indices[polygon_id])
                    print(inside_rects_indices[polygon_id])
                    # for i in intersect_rects_indices[polygon_id]:
                    #     updated_fig.data[i].fillcolor = "rgba(0,120,120,0.5)"
                    # for i in inside_rects_indices[polygon_id]:
                    #     updated_fig.data[i].fillcolor = "rgba(0,255,0,0.5)"
                    return updated_fig, f"Clicked Polygon# {polygon_id}"
        
            return updated_fig, f"Hovered Polygon# {polygon_id}"
        else:
            return dash.no_update

        # hoverData.line.color = "red"
        # print(hoverData)

    return dash.no_update


# @app.callback(Output("graph", "figure"), Input("graph", "clickData"))
# def show_rects_on_click(clickData):
#     if clickData:
#         updated_fig = Patch()
#         updated_fig.data = fig.data
#         polygon_id = clickData["points"][0]["curveNumber"]
#         if polygon_id < 82:
#             for i in intersect_rects_indices[polygon_id]:
                # updated_fig.data[i].fillcolor = "rgba(0,120,120,0.5)"
#             for i in inside_rects_indices[polygon_id]:
                # updated_fig.data[i].fillcolor = "rgba(0,255,0,0.5)"
#             return updated_fig
#         else:
#             return fig
#         # hoverData.line.color = "red"
#         # print(hoverData)

#     else:
#         return fig


# Run the app
app.run_server(debug=True)

[0, 9, 10]
tensor([], size=(0, 1), dtype=torch.int64)
[0, 7, 8]
tensor([], size=(0, 1), dtype=torch.int64)
[0, 8, 9]
tensor([], size=(0, 1), dtype=torch.int64)
[0, 8, 9]
tensor([], size=(0, 1), dtype=torch.int64)
